<a href="https://colab.research.google.com/github/chrispchen/ChenHypoxia/blob/main/PreparePolyATailLengthTable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount and Load libraries

In [1]:
# Mount google drive

from google.colab import drive
drive.mount('/content/drive/')
data_path = '/you/path/'  #change dir to your project folder

Mounted at /content/drive/


In [2]:
# import libraries
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from glob import glob

# Import PolyTailor bams into dataframes

We sequenced two sets of libraries and processed them individually and now need to join them together.

In [3]:

colnames = [
    'read_id',
    'barcode',
    'mapq',
    'filter',
    'pt_length',
    'per_base',
    'primer_end',
    'pt_start',
    'before_pt',
    'pt_seq',
    'transcript_end',
    'distance',
    'chr',
    'strand',
    'isoform_id',
    'gene_id',
    'assignment_type',
    'assignment_event',
    'exons',
    'additional',
    'comment'
]

df_raw_round1 = pd.read_csv(
    "/your/path/tentsupR10.bam.pT.tsv",
    sep="\t",
    names=colnames,
    header=None,
    low_memory=False
)

df_raw_round1 = df_raw_round1.drop(index = 0).reset_index(drop=True)

df_raw_round1.head()


,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,distance,chr,strand,isoform_id,gene_id,assignment_type,assignment_event,exons,additional,comment
0,a4fa9845-b01a-4cc8-ad88-58cbcb8ff9e9,unknown,1,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,b08e7e33-7610-48f3-8ed9-f1a1293ed16f,SQK-NBD114-24_barcode07,60,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,chrUn_KN150342v1,.,TCONS_00129360,LL0000038001,ambiguous,mono_exon_match,39132-39280,gene_assignment=unique; PolyA=False; Classific...,NaN
2,1b514c91-eccd-4fe6-b857-1093e4061fcd,SQK-NBD114-24_barcode11,60,not_continuous,-0.2,3.9,100,101,GAGGCAGAGG,TTTTT,...,NaN,chr1,.,.,.,noninformative,.,2235-3171,Classification=intergenic;,NaN
3,8ab9bff2-43b2-4696-a238-b6c72f444674,SQK-NBD114-24_barcode07,18,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,chr1,.,.,.,noninformative,.,2866-3405,Classification=intergenic;,NaN
4,7f1adb2e-6d1c-483d-8bfc-037643264136,SQK-NBD114-24_barcode08,49,not_continuous,-0.2,2.6,144,149,CAGAGTCTGA,TTTT,...,NaN,chr1,.,.,.,noninformative,.,2877-3141,Classification=intergenic;,NaN


In [4]:

colnames = [
    'read_id',
    'barcode',
    'mapq',
    'filter',
    'pt_length',
    'per_base',
    'primer_end',
    'pt_start',
    'before_pt',
    'pt_seq',
    'transcript_end',
    'distance',
    'chr',
    'strand',
    'isoform_id',
    'gene_id',
    'assignment_type',
    'assignment_event',
    'exons',
    'additional',
    'comment'
]

df_raw_round2 = pd.read_csv(
    "/your/path/tentsupR10resumed2.bam.pT.tsv",
    sep="\t",
    names=colnames,   # force your own header
    header=None,      # don’t trust file’s header
    low_memory=False
)

df_raw_round2 = df_raw_round2.drop(index = 0).reset_index(drop=True)

df_raw_round2.head()


,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,distance,chr,strand,isoform_id,gene_id,assignment_type,assignment_event,exons,additional,comment
0,959decbf-7fcc-4fe6-819a-0c065016c7aa,SQK-NBD114-24_barcode04,60,not_continuous,3.9,2.8,109,112,AGCAGAGTAA,TTTTTTTT,...,NaN,chr1,.,.,.,noninformative,.,1-2538,Classification=intergenic;,NaN
1,87bb8590-9549-4233-9546-8f906bef9217,SQK-NBD114-24_barcode15,24,OK,58.6,2.9,103,103,CAGAGCAGAG,TTTTTTTTTTTTTTTTTTTTTTTTTTTTT,...,NaN,chr1,+,TCONS_00000037,LL0000008435,inconsistent,"alternative_structure_novel:89747-168506,fake_...","46-53,89681-89746,168507-168647",gene_assignment=inconsistent; PolyA=True; Clas...,NaN
2,31d480bf-6875-48a2-b9e5-458bc5964f45,SQK-NBD114-24_barcode14,60,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,chr1,.,.,.,noninformative,.,277-623,Classification=intergenic;,NaN
3,2d6ab212-73c9-4949-80ba-c9ce3741c4f0,SQK-NBD114-24_barcode16,60,no_pT,0,0.0,0,0,NaN,NaN,...,NaN,chr1,.,.,.,noninformative,.,285-2754,Classification=intergenic;,NaN
4,b088328c-2c2d-4694-b7e5-f976ada121aa,SQK-NBD114-24_barcode16,52,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,chr1,.,.,.,noninformative,.,350-1497,Classification=intergenic;,NaN


# Convert transcript nomenclature

In [5]:
# the gtf used (lawson) has an odd transcript nomenclature, need to convert back to normal

lawson = pd.read_excel("/your/path/lawson.xlsx")
lawson

,LLgeneID,ID,EnsdargID,NcbiID,WTkdrl02,WTkdrl04,WTkdrl10,WTneg08,WTneg11,WTneg14,padj,log2FoldChange,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,LL0000000001,a1cf,ENSDARG00000002968,562916.0,31.126931,10.663139,5.783365,13.623193,21.327919,19.336831,0.946989,-0.190788,NaN,NaN,NaN
1,LL0000000002,a2ml,ENSDARG00000056314,100006972.0,0.000000,52.346318,145.548014,22.380960,48.472543,206.904095,0.876781,-0.489054,NaN,NaN,NaN
2,LL0000000003,aaas,ENSDARG00000102274,378454.0,356.014272,496.320641,812.562752,289.006315,534.167429,384.802943,0.586568,0.462857,NaN,kdrlpos-enriched,2141.0
3,LL0000000004,aacs,ENSDARG00000012468,393984.0,225.670249,160.916458,222.659544,448.592293,375.177486,534.663385,0.013545,-1.156847,NaN,kdrlneg-enriched,2136.0
4,LL0000000005,aadac,ENSDARG00000063621,100148912.0,142.989339,74.641971,131.089602,134.285762,127.967515,166.296749,0.772958,-0.297406,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31360,LL0000039745,zwilch,ENSDARG00000057100,336439.0,72.953744,130.865794,255.431945,146.935870,220.065347,182.733056,0.841301,-0.259378,NaN,NaN,NaN
31361,LL0000039746,zyg11,ENSDARG00000007737,555349.0,1044.697617,726.062813,1121.972768,364.906963,517.686764,464.083951,0.013405,1.103023,NaN,NaN,NaN
31362,LL0000039748,zyx,ENSDARG00000075733,564246.0,1964.887511,2502.929483,1987.549698,1208.571861,2596.189426,1832.164764,0.820735,0.195588,NaN,NaN,NaN
31363,LL0000039749,zzef1,ENSDARG00000094380,566682.0,437.722465,571.931989,509.899995,362.960792,434.313989,360.631903,0.494280,0.392135,NaN,NaN,NaN


In [6]:
# Merge old df with lawson on gene_id <-> LLgeneID
df_raw_round1 = df_raw_round1.merge(
    lawson[['LLgeneID', 'ID']],
    left_on='gene_id',
    right_on='LLgeneID',
    how='left'
)

# Rename ID column to Gene
df_raw_round1 = df_raw_round1.rename(columns={'ID': 'Gene'})

# (Optional) Drop LLgeneID if you don't need it anymore
df_raw_round1 = df_raw_round1.drop(columns=['LLgeneID'])



# Merge old df with lawson on gene_id <-> LLgeneID
df_raw_round2 = df_raw_round2.merge(
    lawson[['LLgeneID', 'ID']],
    left_on='gene_id',
    right_on='LLgeneID',
    how='left'
)

# Rename ID column to Gene
df_raw_round2 = df_raw_round2.rename(columns={'ID': 'Gene'})

# (Optional) Drop LLgeneID if you don't need it anymore
df_raw_round2 = df_raw_round2.drop(columns=['LLgeneID'])



# Annotate dataframes

In [7]:
# Define your barcode mapping dictionary

barcode_map = {
    "SQK-NBD114-24_barcode07": "Hyp_rRNA_WT_1a",
    "SQK-NBD114-24_barcode08": "Hyp_rRNA_WT_2a",
    "SQK-NBD114-24_barcode09": "Hyp_rRNA_WT_3a",
    "SQK-NBD114-24_barcode10": "Hyp_rRNA_T42_1a",
    "SQK-NBD114-24_barcode11": "Hyp_rRNA_T42_2a",
    "SQK-NBD114-24_barcode12": "Hyp_rRNA_T42_3a",
    "SQK-NBD114-24_barcode13": "Hyp_rRNA_WT_1",
    "SQK-NBD114-24_barcode14": "Hyp_rRNA_WT_2",
    "SQK-NBD114-24_barcode15": "Hyp_rRNA_WT_3",
    "SQK-NBD114-24_barcode16": "Hyp_rRNA_T42_1",
    "SQK-NBD114-24_barcode17": "Hyp_rRNA_T42_2",
    "SQK-NBD114-24_barcode18": "Hyp_rRNA_T42_3",
    "SQK-NBD114-24_barcode01": "Norm_rRNA_WT_1",
    "SQK-NBD114-24_barcode02": "Norm_rRNA_WT_2",
    "SQK-NBD114-24_barcode03": "Norm_rRNA_WT_3",
    "SQK-NBD114-24_barcode04": "Norm_rRNA_T42_1",
    "SQK-NBD114-24_barcode05": "Norm_rRNA_T42_2",
    "SQK-NBD114-24_barcode06": "Norm_rRNA_T42_3"
}


# Apply mapping to new column
df_raw_round1['Group'] = df_raw_round1['barcode'].map(barcode_map)

# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Wildtype",
    "Hyp_rRNA_WT_2a": "Wildtype",
    "Hyp_rRNA_WT_3a": "Wildtype",
    "Hyp_rRNA_T42_1a": "Tent5ba42",
    "Hyp_rRNA_T42_2a": "Tent5ba42",
    "Hyp_rRNA_T42_3a": "Tent5ba42",
    "Hyp_rRNA_WT_1": "Wildtype",
    "Hyp_rRNA_WT_2": "Wildtype",
    "Hyp_rRNA_WT_3": "Wildtype",
    "Hyp_rRNA_T42_1": "Tent5ba42",
    "Hyp_rRNA_T42_2": "Tent5ba42",
    "Hyp_rRNA_T42_3": "Tent5ba42",
    "Norm_rRNA_WT_1": "Wildtype",
    "Norm_rRNA_WT_2": "Wildtype",
    "Norm_rRNA_WT_3": "Wildtype",
    "Norm_rRNA_T42_1": "Tent5ba42",
    "Norm_rRNA_T42_2": "Tent5ba42",
    "Norm_rRNA_T42_3": "Tent5ba42"
}


# Apply mapping to new column
df_raw_round1['Genotype'] = df_raw_round1['Group'].map(barcode_map)
df_raw_round1


# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Hypoxia18",
    "Hyp_rRNA_WT_2a": "Hypoxia18",
    "Hyp_rRNA_WT_3a": "Hypoxia18",
    "Hyp_rRNA_T42_1a": "Hypoxia18",
    "Hyp_rRNA_T42_2a": "Hypoxia18",
    "Hyp_rRNA_T42_3a": "Hypoxia18",
    "Hyp_rRNA_WT_1": "Hypoxia18",
    "Hyp_rRNA_WT_2": "Hypoxia18",
    "Hyp_rRNA_WT_3": "Hypoxia18",
    "Hyp_rRNA_T42_1": "Hypoxia18",
    "Hyp_rRNA_T42_2": "Hypoxia18",
    "Hyp_rRNA_T42_3": "Hypoxia18",
    "Norm_rRNA_WT_1": "Normoxia6hpf",
    "Norm_rRNA_WT_2": "Normoxia6hpf",
    "Norm_rRNA_WT_3": "Normoxia6hpf",
    "Norm_rRNA_T42_1": "Normoxia6hpf",
    "Norm_rRNA_T42_2": "Normoxia6hpf",
    "Norm_rRNA_T42_3": "Normoxia6hpf"
}


# Apply mapping to new column
df_raw_round1['Condition'] = df_raw_round1['Group'].map(barcode_map)
df_raw_round1


# Apply mapping to new column
df_raw_round1['Method'] = df_raw_round1['Group'].map(barcode_map)
df_raw_round1


# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Batch1",
    "Hyp_rRNA_WT_2a": "Batch1",
    "Hyp_rRNA_WT_3a": "Batch1",
    "Hyp_rRNA_T42_1a": "Batch1",
    "Hyp_rRNA_T42_2a": "Batch1",
    "Hyp_rRNA_T42_3a": "Batch1",
    "Hyp_rRNA_WT_1": "Batch2",
    "Hyp_rRNA_WT_2": "Batch2",
    "Hyp_rRNA_WT_3": "Batch2",
    "Hyp_rRNA_T42_1": "Batch2",
    "Hyp_rRNA_T42_2": "Batch2",
    "Hyp_rRNA_T42_3": "Batch2",
    "Norm_rRNA_WT_1": "Batch2",
    "Norm_rRNA_WT_2": "Batch2",
    "Norm_rRNA_WT_3": "Batch2",
    "Norm_rRNA_T42_1": "Batch2",
    "Norm_rRNA_T42_2": "Batch2",
    "Norm_rRNA_T42_3": "Batch2"
}


# Apply mapping to new column
df_raw_round1['Batch'] = df_raw_round1['Group'].map(barcode_map)
df_raw_round1


,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,assignment_event,exons,additional,comment,Gene,Group,Genotype,Condition,Method,Batch
0,a4fa9845-b01a-4cc8-ad88-58cbcb8ff9e9,unknown,1,no_primer,0,0.0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,b08e7e33-7610-48f3-8ed9-f1a1293ed16f,SQK-NBD114-24_barcode07,60,no_primer,0,0.0,0,0,NaN,NaN,...,mono_exon_match,39132-39280,gene_assignment=unique; PolyA=False; Classific...,NaN,XLOC_043623,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1
2,1b514c91-eccd-4fe6-b857-1093e4061fcd,SQK-NBD114-24_barcode11,60,not_continuous,-0.2,3.9,100,101,GAGGCAGAGG,TTTTT,...,.,2235-3171,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_2a,Tent5ba42,Hypoxia18,Hypoxia18,Batch1
3,8ab9bff2-43b2-4696-a238-b6c72f444674,SQK-NBD114-24_barcode07,18,no_primer,0,0.0,0,0,NaN,NaN,...,.,2866-3405,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1
4,7f1adb2e-6d1c-483d-8bfc-037643264136,SQK-NBD114-24_barcode08,49,not_continuous,-0.2,2.6,144,149,CAGAGTCTGA,TTTT,...,.,2877-3141,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_2a,Wildtype,Hypoxia18,Hypoxia18,Batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4082860,8682a084-01f5-421d-9df3-4ad17b2d1e3a,SQK-NBD114-24_barcode07,60,OK,42.4,2.1,108,108,GCAGAGCAAG,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT,...,"mono_exon_match,tes_match_precise:0,correct_po...",16324-16448,gene_assignment=unique; PolyA=True; Classifica...,NaN,mt-cyb,Hyp_rRNA_WT_1a,Wildtype,Hypoxia18,Hypoxia18,Batch1
4082861,aa57a701-316e-4957-a4f3-b674730070d5,SQK-NBD114-24_barcode11,60,OK,39.9,1.6,101,101,CAGAGCAGAG,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT,...,"mono_exon_match,tes_match_precise:0,correct_po...",16326-16448,gene_assignment=unique; PolyA=True; Classifica...,NaN,mt-cyb,Hyp_rRNA_T42_2a,Tent5ba42,Hypoxia18,Hypoxia18,Batch1
4082862,526ab2d9-1c67-4aff-8e52-1ad02f05ee6e,unknown,60,no_primer,0,0.0,0,0,NaN,NaN,...,"mono_exon_match,tes_match:46,exon_elongation_3:46",16344-16494,gene_assignment=unique_minor_difference; PolyA...,NaN,mt-cyb,NaN,NaN,NaN,NaN,NaN
4082863,294ff7fa-6a21-48a0-855f-a7b59a0707f4,SQK-NBD114-24_barcode09,60,OK,38.1,2.3,108,108,CAGAGCAGAG,TTTTTTTTTTTTTTTTATTTTTTTTTTTTTT,...,"mono_exon_match,tes_match_precise:0,correct_po...",16346-16448,gene_assignment=unique; PolyA=True; Classifica...,NaN,mt-cyb,Hyp_rRNA_WT_3a,Wildtype,Hypoxia18,Hypoxia18,Batch1


In [8]:
df_raw_round1.barcode.unique()

array(['unknown', 'SQK-NBD114-24_barcode07', 'SQK-NBD114-24_barcode11',
       'SQK-NBD114-24_barcode08', 'SQK-NBD114-24_barcode12',
       'SQK-NBD114-24_barcode09', 'SQK-NBD114-24_barcode10',
       'SQK-NBD114-24_barcode18', 'SQK-NBD114-24_barcode05',
       'SQK-NBD114-24_barcode02', 'SQK-NBD114-24_barcode13',
       'SQK-NBD114-24_barcode15', 'SQK-NBD114-24_barcode19',
       'SQK-NBD114-24_barcode01'], dtype=object)

In [9]:
df_raw_round2.barcode.unique()

array(['SQK-NBD114-24_barcode04', 'SQK-NBD114-24_barcode15',
       'SQK-NBD114-24_barcode14', 'SQK-NBD114-24_barcode16',
       'SQK-NBD114-24_barcode17', 'SQK-NBD114-24_barcode05',
       'SQK-NBD114-24_barcode13', 'unknown', 'SQK-NBD114-24_barcode03',
       'SQK-NBD114-24_barcode21', 'SQK-NBD114-24_barcode06',
       'SQK-NBD114-24_barcode18', 'SQK-NBD114-24_barcode01',
       'SQK-NBD114-24_barcode02', 'SQK-NBD114-24_barcode19',
       'SQK-NBD114-24_barcode20', 'SQK-NBD114-24_barcode22',
       'SQK-NBD114-24_barcode24', 'SQK-NBD114-24_barcode23',
       'SQK-NBD114-24_barcode12', 'SQK-NBD114-24_barcode11',
       'SQK-NBD114-24_barcode10', 'SQK-NBD114-24_barcode09',
       'SQK-NBD114-24_barcode07', 'SQK-NBD114-24_barcode08'], dtype=object)

In [10]:
# Define your barcode mapping dictionary

barcode_map = {
    "SQK-NBD114-24_barcode07": "Hyp_rRNA_WT_1a",
    "SQK-NBD114-24_barcode08": "Hyp_rRNA_WT_2a",
    "SQK-NBD114-24_barcode09": "Hyp_rRNA_WT_3a",
    "SQK-NBD114-24_barcode10": "Hyp_rRNA_T42_1a",
    "SQK-NBD114-24_barcode11": "Hyp_rRNA_T42_2a",
    "SQK-NBD114-24_barcode12": "Hyp_rRNA_T42_3a",
    "SQK-NBD114-24_barcode13": "Hyp_rRNA_WT_1",
    "SQK-NBD114-24_barcode14": "Hyp_rRNA_WT_2",
    "SQK-NBD114-24_barcode15": "Hyp_rRNA_WT_3",
    "SQK-NBD114-24_barcode16": "Hyp_rRNA_T42_1",
    "SQK-NBD114-24_barcode17": "Hyp_rRNA_T42_2",
    "SQK-NBD114-24_barcode18": "Hyp_rRNA_T42_3",
    "SQK-NBD114-24_barcode01": "Norm_rRNA_WT_1",
    "SQK-NBD114-24_barcode02": "Norm_rRNA_WT_2",
    "SQK-NBD114-24_barcode03": "Norm_rRNA_WT_3",
    "SQK-NBD114-24_barcode04": "Norm_rRNA_T42_1",
    "SQK-NBD114-24_barcode05": "Norm_rRNA_T42_2",
    "SQK-NBD114-24_barcode06": "Norm_rRNA_T42_3"
}


# Apply mapping to new column
df_raw_round2['Group'] = df_raw_round2['barcode'].map(barcode_map)

# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Wildtype",
    "Hyp_rRNA_WT_2a": "Wildtype",
    "Hyp_rRNA_WT_3a": "Wildtype",
    "Hyp_rRNA_T42_1a": "Tent5ba42",
    "Hyp_rRNA_T42_2a": "Tent5ba42",
    "Hyp_rRNA_T42_3a": "Tent5ba42",
    "Hyp_rRNA_WT_1": "Wildtype",
    "Hyp_rRNA_WT_2": "Wildtype",
    "Hyp_rRNA_WT_3": "Wildtype",
    "Hyp_rRNA_T42_1": "Tent5ba42",
    "Hyp_rRNA_T42_2": "Tent5ba42",
    "Hyp_rRNA_T42_3": "Tent5ba42",
    "Norm_rRNA_WT_1": "Wildtype",
    "Norm_rRNA_WT_2": "Wildtype",
    "Norm_rRNA_WT_3": "Wildtype",
    "Norm_rRNA_T42_1": "Tent5ba42",
    "Norm_rRNA_T42_2": "Tent5ba42",
    "Norm_rRNA_T42_3": "Tent5ba42"
}


# Apply mapping to new column
df_raw_round2['Genotype'] = df_raw_round2['Group'].map(barcode_map)
df_raw_round2


# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Hypoxia18",
    "Hyp_rRNA_WT_2a": "Hypoxia18",
    "Hyp_rRNA_WT_3a": "Hypoxia18",
    "Hyp_rRNA_T42_1a": "Hypoxia18",
    "Hyp_rRNA_T42_2a": "Hypoxia18",
    "Hyp_rRNA_T42_3a": "Hypoxia18",
    "Hyp_rRNA_WT_1": "Hypoxia18",
    "Hyp_rRNA_WT_2": "Hypoxia18",
    "Hyp_rRNA_WT_3": "Hypoxia18",
    "Hyp_rRNA_T42_1": "Hypoxia18",
    "Hyp_rRNA_T42_2": "Hypoxia18",
    "Hyp_rRNA_T42_3": "Hypoxia18",
    "Norm_rRNA_WT_1": "Normoxia6hpf",
    "Norm_rRNA_WT_2": "Normoxia6hpf",
    "Norm_rRNA_WT_3": "Normoxia6hpf",
    "Norm_rRNA_T42_1": "Normoxia6hpf",
    "Norm_rRNA_T42_2": "Normoxia6hpf",
    "Norm_rRNA_T42_3": "Normoxia6hpf"
}


# Apply mapping to new column
df_raw_round2['Condition'] = df_raw_round2['Group'].map(barcode_map)
df_raw_round2


# Apply mapping to new column
df_raw_round2['Method'] = df_raw_round2['Group'].map(barcode_map)
df_raw_round2


# Define your barcode mapping dictionary
barcode_map = {
    "Hyp_rRNA_WT_1a": "Batch1",
    "Hyp_rRNA_WT_2a": "Batch1",
    "Hyp_rRNA_WT_3a": "Batch1",
    "Hyp_rRNA_T42_1a": "Batch1",
    "Hyp_rRNA_T42_2a": "Batch1",
    "Hyp_rRNA_T42_3a": "Batch1",
    "Hyp_rRNA_WT_1": "Batch2",
    "Hyp_rRNA_WT_2": "Batch2",
    "Hyp_rRNA_WT_3": "Batch2",
    "Hyp_rRNA_T42_1": "Batch2",
    "Hyp_rRNA_T42_2": "Batch2",
    "Hyp_rRNA_T42_3": "Batch2",
    "Norm_rRNA_WT_1": "Batch2",
    "Norm_rRNA_WT_2": "Batch2",
    "Norm_rRNA_WT_3": "Batch2",
    "Norm_rRNA_T42_1": "Batch2",
    "Norm_rRNA_T42_2": "Batch2",
    "Norm_rRNA_T42_3": "Batch2"
}


# Apply mapping to new column
df_raw_round2['Batch'] = df_raw_round2['Group'].map(barcode_map)
df_raw_round2

,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,assignment_event,exons,additional,comment,Gene,Group,Genotype,Condition,Method,Batch
0,959decbf-7fcc-4fe6-819a-0c065016c7aa,SQK-NBD114-24_barcode04,60,not_continuous,3.9,2.8,109,112,AGCAGAGTAA,TTTTTTTT,...,.,1-2538,Classification=intergenic;,NaN,NaN,Norm_rRNA_T42_1,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2
1,87bb8590-9549-4233-9546-8f906bef9217,SQK-NBD114-24_barcode15,24,OK,58.6,2.9,103,103,CAGAGCAGAG,TTTTTTTTTTTTTTTTTTTTTTTTTTTTT,...,"alternative_structure_novel:89747-168506,fake_...","46-53,89681-89746,168507-168647",gene_assignment=inconsistent; PolyA=True; Clas...,NaN,cul4a,Hyp_rRNA_WT_3,Wildtype,Hypoxia18,Hypoxia18,Batch2
2,31d480bf-6875-48a2-b9e5-458bc5964f45,SQK-NBD114-24_barcode14,60,no_primer,0,0.0,0,0,NaN,NaN,...,.,277-623,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_2,Wildtype,Hypoxia18,Hypoxia18,Batch2
3,2d6ab212-73c9-4949-80ba-c9ce3741c4f0,SQK-NBD114-24_barcode16,60,no_pT,0,0.0,0,0,NaN,NaN,...,.,285-2754,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2
4,b088328c-2c2d-4694-b7e5-f976ada121aa,SQK-NBD114-24_barcode16,52,no_primer,0,0.0,0,0,NaN,NaN,...,.,350-1497,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24939884,eddc7366-6312-47a7-b918-5beedcc5abe7,SQK-NBD114-24_barcode13,60,no_primer,0,0.0,0,0,NaN,NaN,...,"mono_exon_match,terminal_position_match:50,exo...",16399-16596,gene_assignment=inconsistent_non_intronic; Pol...,NaN,NaN,Hyp_rRNA_WT_1,Wildtype,Hypoxia18,Hypoxia18,Batch2
24939885,cdbb438e-243c-4122-a166-959b0145e540,SQK-NBD114-24_barcode05,60,OK,10.8,3.5,104,104,CAGAGCAGAG,TTTTTTTTTTTTTTTTTATT,...,"mono_exon_match,tss_match:49,exon_elongation_5...",16400-16515,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_2,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2
24939886,a25e139b-ff92-4766-8e3b-86a37959dea8,SQK-NBD114-24_barcode16,60,no_primer,0,0.0,0,0,NaN,NaN,...,"mono_exon_match,tss_match:47,exon_elongation_5...",16402-16560,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2
24939887,4d397346-f635-44df-b346-b6588a666674,SQK-NBD114-24_barcode04,60,OK,12.9,2.9,103,103,CAGAGCAGAG,TTATTTTTTTTTTTTTT,...,"mono_exon_match,tss_match:46,exon_elongation_5...",16403-16522,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_1,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2


# Normalize polyT lengths

In [11]:
# the nano3p seq documentation says:
# "PolyTailor underestimates tail length, therefore we normalise it by a factor a optionally adding b." from
# https://github.com/novoalab/polyTailor/blob/795c3f132c01507769a24348c0c55a35f5ef2045/notebook/DNA_standards.ipynb#L70
# where a, b = 1.25, 2

y = "pt_length"
a, b = 1.25, 2

df_raw_round1[y] = pd.to_numeric(df_raw_round1[y], errors="coerce")  # turns non-numeric into NaN


df_raw_round1['Adjusted_pt_length'] = df_raw_round1[y] * a + b
df_raw_round1


y = "pt_length"
a, b = 1.25, 2

df_raw_round2[y] = pd.to_numeric(df_raw_round2[y], errors="coerce")  # turns non-numeric into NaN


df_raw_round2['Adjusted_pt_length'] = df_raw_round2[y] * a + b
df_raw_round2

,read_id,barcode,mapq,filter,pt_length,per_base,primer_end,pt_start,before_pt,pt_seq,...,exons,additional,comment,Gene,Group,Genotype,Condition,Method,Batch,Adjusted_pt_length
0,959decbf-7fcc-4fe6-819a-0c065016c7aa,SQK-NBD114-24_barcode04,60,not_continuous,3.9,2.8,109,112,AGCAGAGTAA,TTTTTTTT,...,1-2538,Classification=intergenic;,NaN,NaN,Norm_rRNA_T42_1,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2,6.875
1,87bb8590-9549-4233-9546-8f906bef9217,SQK-NBD114-24_barcode15,24,OK,58.6,2.9,103,103,CAGAGCAGAG,TTTTTTTTTTTTTTTTTTTTTTTTTTTTT,...,"46-53,89681-89746,168507-168647",gene_assignment=inconsistent; PolyA=True; Clas...,NaN,cul4a,Hyp_rRNA_WT_3,Wildtype,Hypoxia18,Hypoxia18,Batch2,75.250
2,31d480bf-6875-48a2-b9e5-458bc5964f45,SQK-NBD114-24_barcode14,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,277-623,Classification=intergenic;,NaN,NaN,Hyp_rRNA_WT_2,Wildtype,Hypoxia18,Hypoxia18,Batch2,2.000
3,2d6ab212-73c9-4949-80ba-c9ce3741c4f0,SQK-NBD114-24_barcode16,60,no_pT,0.0,0.0,0,0,NaN,NaN,...,285-2754,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2,2.000
4,b088328c-2c2d-4694-b7e5-f976ada121aa,SQK-NBD114-24_barcode16,52,no_primer,0.0,0.0,0,0,NaN,NaN,...,350-1497,Classification=intergenic;,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2,2.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24939884,eddc7366-6312-47a7-b918-5beedcc5abe7,SQK-NBD114-24_barcode13,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,16399-16596,gene_assignment=inconsistent_non_intronic; Pol...,NaN,NaN,Hyp_rRNA_WT_1,Wildtype,Hypoxia18,Hypoxia18,Batch2,2.000
24939885,cdbb438e-243c-4122-a166-959b0145e540,SQK-NBD114-24_barcode05,60,OK,10.8,3.5,104,104,CAGAGCAGAG,TTTTTTTTTTTTTTTTTATT,...,16400-16515,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_2,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2,15.500
24939886,a25e139b-ff92-4766-8e3b-86a37959dea8,SQK-NBD114-24_barcode16,60,no_primer,0.0,0.0,0,0,NaN,NaN,...,16402-16560,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Hyp_rRNA_T42_1,Tent5ba42,Hypoxia18,Hypoxia18,Batch2,2.000
24939887,4d397346-f635-44df-b346-b6588a666674,SQK-NBD114-24_barcode04,60,OK,12.9,2.9,103,103,CAGAGCAGAG,TTATTTTTTTTTTTTTT,...,16403-16522,gene_assignment=unique_minor_difference; PolyA...,NaN,NaN,Norm_rRNA_T42_1,Tent5ba42,Normoxia6hpf,Normoxia6hpf,Batch2,18.125


# Create joined object

In [12]:
# Create joined object
## filter out impossible barcodes

possible1 = ["SQK-NBD114-24_barcode07",
    "SQK-NBD114-24_barcode08",
    "SQK-NBD114-24_barcode09",
    "SQK-NBD114-24_barcode10",
    "SQK-NBD114-24_barcode11",
    "SQK-NBD114-24_barcode12"]

df_raw_round1a = df_raw_round1.loc[df_raw_round1.barcode.isin(possible1)]


possible2 = [ "SQK-NBD114-24_barcode13",
    "SQK-NBD114-24_barcode14",
    "SQK-NBD114-24_barcode15",
    "SQK-NBD114-24_barcode16",
    "SQK-NBD114-24_barcode17",
    "SQK-NBD114-24_barcode18",
    "SQK-NBD114-24_barcode01",
    "SQK-NBD114-24_barcode02",
    "SQK-NBD114-24_barcode03",
    "SQK-NBD114-24_barcode04",
    "SQK-NBD114-24_barcode05",
    "SQK-NBD114-24_barcode06"]

df_raw_round2a = df_raw_round2.loc[df_raw_round2.barcode.isin(possible2)]

# Export csv

In [13]:
# Combine them row-wise
combined = pd.concat([df_raw_round1a, df_raw_round2a], ignore_index=True)

# Save the result
combined.to_csv("/your/path/CombinedR1R2_PolyAtable_annotated.csv", sep="\t", index=False)

